In [ ]:
!pip install transformers torch

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

model_name = "microsoft/DialoGPT-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
model.to(device)

In [ ]:
def generate_response(user_input, chat_history_ids=None):
    new_input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors='pt').to(device)

    if chat_history_ids is not None:
        bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
    else:
        bot_input_ids = new_input_ids

    chat_history_ids = model.generate(
        bot_input_ids,
        max_new_tokens=100,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        temperature=0.7,
        top_k=50,
        top_p=0.9,
        no_repeat_ngram_size=3
    )

    response = tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True
    )

    return response, chat_history_ids

In [ ]:
def smart_reply(user_input):
    text = user_input.lower()

    if "artificial intelligence" in text or "ai" in text:
        return "Artificial Intelligence is the simulation of human intelligence in machines."
    elif "machine learning" in text:
        return "Machine Learning is a subset of AI that learns from data."
    elif "python" in text:
        return "Python was created by Guido van Rossum in 1991."

    return None

In [ ]:
print("Chatbot: Hello! Type 'exit' to stop")

chat_history = None

while True:
    user_input = input("User: ")

    if user_input.lower() in ["exit", "quit"]:
        print("Chatbot: Goodbye!")
        break

    fallback = smart_reply(user_input)

    if fallback:
        print("Chatbot:", fallback)
    else:
        response, chat_history = generate_response(user_input, chat_history)
        print("Chatbot:", response)